# Evidencia de Aprendizaje 3 — Optimización de procesos de desarrollo
### Automatización con GitHub Actions y despliegue de un servicio en la nube

**Jeremy Iván Pedraza Hernández** · Programación para Análisis de Datos · IU Digital de Antioquia

- Repositorio: https://github.com/Jeremy-Pedraza/Programacion-Analisis-Datos
- Aplicación web desplegada (Render): https://galaxy-a55-analytics.onrender.com

Este cuaderno es un artefacto **ejecutable**: en lugar de describir el proyecto, lo pone en marcha. Recorre el historial Git, ejecuta las pruebas, arranca la aplicación web y consulta sus *endpoints*, y muestra los *workflows* de CI/CD. Funciona tanto en local (dentro del repositorio) como en Google Colab (clona el repositorio automáticamente).

## Fase 0 — Configuración del entorno

Detecta si el cuaderno se ejecuta dentro del repositorio o en Colab; en Colab clona el repositorio. Luego instala el paquete y sus dependencias de desarrollo.

In [ ]:
import os, subprocess, sys
from pathlib import Path

def encontrar_raiz():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'ml_scraper' / 'web.py').exists():
            return cand
    return None

RAIZ = encontrar_raiz()
if RAIZ is None:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Jeremy-Pedraza/Programacion-Analisis-Datos.git'], check=True)
    RAIZ = Path.cwd() / 'Programacion-Analisis-Datos'
os.chdir(RAIZ)
print('Raíz del proyecto:', RAIZ)

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements-dev.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Dependencias instaladas.')

## Fase 1 — Control de versiones (Git)

El proyecto se construyó con un modelo de ramas (`main`, `develop`, `feature/*`, `docs/*`) e integraciones por *pull request*. Las celdas muestran el grafo de commits y la convención *Conventional Commits*.

In [ ]:
import subprocess
def git(*args):
    return subprocess.run(['git', *args], capture_output=True, text=True).stdout
print(git('log', '--oneline', '--graph', '--all', '-20'))

In [ ]:
print('Ramas:'); print(git('branch', '-a'))
print('Etiquetas:'); print(git('tag'))

## Fase 2 — Pruebas automatizadas (replica la integración continua)

Las mismas 37 pruebas que ejecuta el *workflow* `ci.yml` en cada *push* y *pull request*, con cobertura. Son la puerta de calidad que debe pasar antes de desplegar.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '--cov', '--cov-report=term-missing'],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print('RUFF:'); print(subprocess.run([sys.executable, '-m', 'ruff', 'check', '.'],
                                     capture_output=True, text=True).stdout or 'All checks passed!')

## Fase 3 — La aplicación web en marcha

Se arranca la aplicación FastAPI **en proceso** con `TestClient` (sin servidor externo) y se consultan sus *endpoints*: el estado del servicio, los indicadores de posicionamiento y la ficha del producto. Es el mismo código que se despliega en la nube.

In [ ]:
from fastapi.testclient import TestClient
from ml_scraper.web import app
cliente = TestClient(app)
print('GET /health ->', cliente.get('/health').json())

In [ ]:
import pandas as pd
ind = cliente.get('/api/indicadores').json()
pd.DataFrame(ind.items(), columns=['Indicador', 'Valor'])

In [ ]:
comp = pd.DataFrame(cliente.get('/api/competencia').json())
print('Competidores:', len(comp))
comp[['titulo', 'precio', 'calificacion']].sort_values('precio').head(12)

In [ ]:
# Regresión de negocio: la conclusión de la EA1 debe mantenerse en la API desplegada.
assert abs(ind['Desviación vs. promedio (%)'] - 17.05) < 0.01
print('OK — el artículo se ofrece', ind['Desviación vs. promedio (%)'], '% por encima del promedio.')

## Fase 4 — Automatización y despliegue (GitHub Actions + Render)

Los tres *workflows* del repositorio: integración continua (`ci.yml`), despliegue del reporte estático (`cd.yml`) y **despliegue del servicio en la nube** (`cd-render.yml`). Este último encadena: pruebas → build de la imagen Docker → *Deploy Hook* de Render → prueba de humo.

In [ ]:
from pathlib import Path
wf = Path('.github/workflows')
for f in ['ci.yml', 'cd.yml', 'cd-render.yml']:
    print('='*70); print(f); print('='*70)
    print((wf / f).read_text(encoding='utf-8'))

In [ ]:
print('Dockerfile:'); print(Path('Dockerfile').read_text(encoding='utf-8'))
print('render.yaml:'); print(Path('render.yaml').read_text(encoding='utf-8'))

La aplicación está desplegada en **https://galaxy-a55-analytics.onrender.com**. Si el servicio está activo, la siguiente celda consulta su estado en producción (en Colab con red). En el plan gratuito el primer acceso puede tardar unos segundos por el arranque en frío.

In [ ]:
import urllib.request, json
try:
    with urllib.request.urlopen('https://galaxy-a55-analytics.onrender.com/health', timeout=60) as r:
        print('Producción /health ->', json.load(r))
except Exception as e:
    print('No se pudo consultar producción (servicio dormido o sin red):', e)

## Fase 5 — Conclusiones

- El análisis dejó de ser un reporte estático y se convirtió en un **servicio web** con API REST.
- La **contenerización** garantiza que lo probado es lo desplegado; GitHub Actions es el único origen de los despliegues.
- El pipeline `test → docker → deploy → smoke` solo publica lo que superó todas las verificaciones, y la prueba de humo confirma que el servicio quedó en línea.
- La **regresión de negocio** (+17,05 %) se mantiene verificable en la API desplegada.
- Se completó el ciclo DevOps de las tres evidencias: extracción (EA1) → CI/CD y software versionado (EA2) → despliegue de un servicio en la nube (EA3).